In [1]:
!pip install scikeras
!pip install imbalanced-learn xgboost lightgbm catboost
!pip install scikit-learn==1.5.2 imbalanced-learn==0.12.3 scikeras==0.13.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.3/258.3 kB 16.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [2]:
# ==============================================================================
# 🗺️ ESTIMASI PRODUKSI PADI - ARSITEKTUR MURNI SESUAI PAPER (T4x2 GPU)
# MODEL: LSTM / BI-LSTM (122 -> 61 NEURONS) | BATCH 128 | LR 0.01 | EPOCHS 5000
# DATASET: unik_per_bulan.csv (Proporsi Non-Padi Imbang)
# ==============================================================================

import pandas as pd
import numpy as np
import os, gc, time, joblib, warnings
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Reshape, Input, Dropout, Concatenate, Lambda
from tensorflow.keras.layers import LSTM as KerasLSTM
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Preprocessing & Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             precision_score, recall_score, confusion_matrix)
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')

# ==========================================================
# 🎯 0. CONTROL PANEL (GANTI MODEL DI SINI)
# ==========================================================
MODEL_PILIHAN = 'LSTM'   # Pilihan: 'LSTM' atau 'BiLSTM'

print(f"🔥 MEMULAI NOTEBOOK UNTUK: {MODEL_PILIHAN.upper()} (PURE PAPER ARCHITECTURE) 🔥\n")

# ==========================================================
# 🔍 1. PEMUATAN DATASET
# ==========================================================
dataset_path = '/kaggle/input/datasets/jejehayu/data-klasifikasi-padi/unik_per_bulan.csv'
print(f"🔍 Membaca dataset dari: {dataset_path}")

if os.path.exists(dataset_path):
    kolom_input = ['id', 'longitude', 'latitude', 'Tahun', 'Bulan', 'date_img', 'Lahan',
                   'VV', 'VH', 'RVI', 'NDVI', 'LSWI', 'EVI', 'BSI']
    df = pd.read_csv(dataset_path, names=kolom_input, header=0)
    df['date_img'] = pd.to_datetime(df['date_img'])
    df = df[df['Tahun'] != 2021].reset_index(drop=True)
    print(f"✅ Dataset berhasil dimuat! Jumlah baris: {len(df)}")
else:
    raise FileNotFoundError(f"❌ ERROR: File {dataset_path} tidak ditemukan.")

# ==========================================================
# ⚙️ 2. FEATURE ENGINEERING (WAKTU & LAG SATELIT)
# ==========================================================
df['segmen'] = df['longitude'].astype(str) + '_' + df['latitude'].astype(str)
df = df.sort_values(['segmen', 'Tahun', 'Bulan']).reset_index(drop=True)

df['Bulan_sin'] = np.sin(2 * np.pi * df['Bulan'] / 12)
df['Bulan_cos'] = np.cos(2 * np.pi * df['Bulan'] / 12)

features_sat = ['VV', 'VH', 'RVI', 'NDVI', 'LSWI', 'EVI', 'BSI']
for lag in [1, 2, 3]:
    for feat in features_sat:
        df[f'{feat}_lag{lag}'] = df.groupby('segmen')[feat].shift(lag)

df_panel = df.dropna(subset=['Lahan']).reset_index(drop=True)

# ==========================================================
# ⚖️ 3. BALANCING DATA (PROPORSI IMBANG)
# ==========================================================
print("⚖️ Melakukan penyeimbangan data dinamis...")
df_panel['Lahan'] = df_panel['Lahan'].astype(str).str.strip().str.lower()

df_padi = df_panel[df_panel['Lahan'] == 'padi']
n_padi = len(df_padi)
kelas_non_padi = df_panel[df_panel['Lahan'] != 'padi']['Lahan'].unique()
k_classes = len(kelas_non_padi)

if k_classes > 0:
    kuota_per_kelas = n_padi // k_classes
    sisa = n_padi % k_classes
    df_non_padi_terpilih = []
    
    for i, cls in enumerate(kelas_non_padi):
        df_cls = df_panel[df_panel['Lahan'] == cls]
        target_n = kuota_per_kelas + (1 if i < sisa else 0)
        n_ambil = min(target_n, len(df_cls))
        df_non_padi_terpilih.append(df_cls.sample(n=n_ambil, random_state=42))
        
    df_non_padi_balanced = pd.concat(df_non_padi_terpilih)
    df_balanced = pd.concat([df_padi, df_non_padi_balanced])
else:
    df_balanced = df_padi

df_balanced['Label_Biner'] = np.where(df_balanced['Lahan'] == 'padi', 1, 0)
df_panel = df_balanced.sort_values(['Tahun', 'Bulan', 'segmen']).reset_index(drop=True)

# Menyusun kolom input
kolom_temporal = []
for lag in [3, 2, 1]:
    for feat in features_sat:
        kolom_temporal.append(f'{feat}_lag{lag}')
for feat in features_sat:
    kolom_temporal.append(feat)

spasial = ['Bulan_sin', 'Bulan_cos', 'Tahun']
feats_all = kolom_temporal + spasial

X = df_panel[feats_all].astype(np.float32)
y = df_panel['Label_Biner']

# Memisahkan data train dan test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 1. TAMBAHAN BARU: Menambal nilai NaN akibat proses shift/lag
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# 2. Scaling data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ==========================================================
# 🧠 4. DEFINISI ARSITEKTUR MURNI PAPER (T4x2 GPU)
# ==========================================================
N_SAT_FEATURES = len(features_sat) # 7
N_TIMESTEP = 4
N_TEMPORAL_COLS = N_SAT_FEATURES * N_TIMESTEP 

# Strategi Multi-GPU
strategy = tf.distribute.MirroredStrategy()
print(f"🚀 Jumlah GPU yang tersinkronisasi: {strategy.num_replicas_in_sync}")

with strategy.scope():
    inputs = Input(shape=(X_train.shape[1],), name='input_gabungan')
    
    x_sat = Lambda(lambda z: z[:, :N_TEMPORAL_COLS], name='slice_satelit')(inputs)
    x_time = Lambda(lambda z: z[:, N_TEMPORAL_COLS:], name='slice_waktu')(inputs)
    x_sat_reshaped = Reshape((N_TIMESTEP, N_SAT_FEATURES), name='reshape_3d')(x_sat)

    # 1. LAYER PERTAMA: 122 Neuron
    if MODEL_PILIHAN.upper() == 'BILSTM':
        rnn_1 = Bidirectional(KerasLSTM(122, return_sequences=True), name='bilstm_122')(x_sat_reshaped)
        # 2. LAYER KEDUA: 61 Neuron
        rnn_2 = Bidirectional(KerasLSTM(61, return_sequences=False), name='bilstm_61')(rnn_1)
    else:
        rnn_1 = KerasLSTM(122, return_sequences=True, name='lstm_122')(x_sat_reshaped)
        rnn_2 = KerasLSTM(61, return_sequences=False, name='lstm_61')(rnn_1)

    # 3. DROPOUT 0.2 Sesuai paper
    drop_layer = Dropout(0.2, name='dropout_0.2')(rnn_2)
    
    # Menggabungkan waktu & langsung ke output (tanpa deep fusion)
    merged = Concatenate(name='gabung_fitur')([drop_layer, x_time])
    
    # 4. OUTPUT LAYER 
    # (Di paper pakai Softmax 7 output, kita sesuaikan jadi Sigmoid 1 output karena klasifikasi Biner)
    outputs = Dense(1, activation='sigmoid', name='output')(merged)

    model = Model(inputs=inputs, outputs=outputs)
    
    # 5. OPTIMIZER: Adam, LR 0.01 Sesuai paper
    model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# ==========================================================
# 🚀 5. TRAINING MODEL STATIC (TANPA HYPERPARAMETER TUNING)
# ==========================================================
# Skenario Paper: 5000 Epochs, membagi LR saat training berlangsung
# Kita gunakan ReduceLROnPlateau untuk membagi LR secara dinamis
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=15, min_lr=1e-5, verbose=1)

# EarlyStopping agar Kaggle tidak timeout jika model sudah mencapai performa maksimal (opsional)
early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True, verbose=1)

print(f"\n🚀 Memulai Training {MODEL_PILIHAN} Murni (Epoch=5000, Batch=128, LR=0.01)...")
start_time = time.time()

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=5000,
    batch_size=128,  # Batch size 128 sesuai paper
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ==========================================================
# 📊 6. EVALUASI & PENYIMPANAN
# ==========================================================
# Prediksi probabilitas dan konversi ke label diskrit (0 atau 1)
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
kappa = cohen_kappa_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='macro')
rec = recall_score(y_test, y_pred, average='macro')
waktu = time.time() - start_time

print(f"\n✅ TRAINING {MODEL_PILIHAN} SELESAI!")
print(f"📊 F1-Macro: {f1:.4f} | Acc: {acc:.4f} | Kappa: {kappa:.4f}")
print(f"⏱️ Waktu: {waktu:.1f} detik")

nama_bersih = f"{MODEL_PILIHAN}_PurePaper".replace(" ", "")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')
plt.title(f'Confusion Matrix - {MODEL_PILIHAN} (Pure Paper)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.savefig(f'/kaggle/working/CM_{nama_bersih}.png')
plt.close()

file_hasil = f'/kaggle/working/Summary_Models.csv'
log_data = {
    'Model': f"{MODEL_PILIHAN} (Pure Paper)",
    'Accuracy': acc,
    'F1_Macro': f1,
    'Kappa': kappa,
    'Precision_Macro': prec,
    'Recall_Macro': rec,
    'Waktu_Sekon': round(waktu, 2),
    'Best_Params': "Fixed (122-61, LR=0.01, Batch=128, Drop=0.2)"
}

if os.path.exists(file_hasil):
    df_log = pd.read_csv(file_hasil)
    df_log = pd.concat([df_log, pd.DataFrame([log_data])], ignore_index=True)
else:
    df_log = pd.DataFrame([log_data])
df_log.to_csv(file_hasil, index=False)

joblib.dump(scaler, f'/kaggle/working/Scaler_{nama_bersih}.pkl')
model.save(f'/kaggle/working/Model_{nama_bersih}_Final.h5')
gc.collect()

print(f"📂 Model (.h5) dan Scaler (.pkl) telah disimpan di folder output Kaggle!")

🔥 MEMULAI NOTEBOOK UNTUK: LSTM (PURE PAPER ARCHITECTURE) 🔥

🔍 Membaca dataset dari: /kaggle/input/datasets/jejehayu/data-klasifikasi-padi/unik_per_bulan.csv
✅ Dataset berhasil dimuat! Jumlah baris: 151074
⚖️ Melakukan penyeimbangan data dinamis...
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1789229309.760635      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789229309.763689      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


🚀 Jumlah GPU yang tersinkronisasi: 2

🚀 Memulai Training LSTM Murni (Epoch=5000, Batch=128, LR=0.01)...
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:

📂 Model (.h5) dan Scaler (.pkl) telah disimpan di folder output Kaggle!
